In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
# Caltech-101 (already unzipped) -> folder '101_ObjectCategories'
# 50% train, 25% validation, 25% test
train_ds = tf.keras.utils.image_dataset_from_directory(
    '101_ObjectCategories', validation_split=0.5, subset='training', seed=42,
    image_size=(224, 224), batch_size=64, label_mode='categorical')
rest_ds = tf.keras.utils.image_dataset_from_directory(
    '101_ObjectCategories', validation_split=0.5, subset='validation', seed=42,
    image_size=(224, 224), batch_size=64, label_mode='categorical')

num_classes = len(train_ds.class_names)
half = rest_ds.cardinality().numpy() // 2
val_ds = rest_ds.take(half)
test_ds = rest_ds.skip(half)

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
val_ds = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y))

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False,
                   input_shape=(224, 224, 3))
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
model = models.Sequential()
model.add(base_model)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(num_classes, activation='softmax'))

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
print(model.summary())

In [ ]:
history = model.fit(train_ds,
                    epochs=5,
                    validation_data=val_ds)

In [ ]:
for layer in base_model.layers[-4:]:  # last 4 conv layers
    layer.trainable = True

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # lower LR
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
history_fine = model.fit(train_ds,
                         epochs=3,
                         validation_data=val_ds)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy after fine-tuning:", test_acc)